# Tarefa 2 — Baseline por fluxo e rotulagem do Período B

**Versão refeita para evitar erros em cascata.** Este notebook procura a raiz do projeto automaticamente, tenta localizar os arquivos produzidos pela Tarefa 1 e interrompe cada etapa com uma mensagem clara quando os pré-requisitos não existem.

## Objetivo
1. Auditar o bruto sem modificá-lo.
2. Estimar uma ficha de baseline usando **somente o Período A**.
3. Calcular métricas relativas e classes no **Período B**.
4. Criar um recorte temporal dentro de B: 50% treino, 20% validação e 30% teste.
5. Exportar os artefatos em `data/processed/`.

**Importante:** este notebook não cria dados artificiais e não inventa limites de datas. Se a Tarefa 1 ainda não gerou os arquivos necessários, a execução vai explicar o que está faltando sem causar `NameError` nas células seguintes.

## 0. Imports e descoberta da raiz do projeto

O notebook está em `documentos/notebooks/Arvore`, então não devemos assumir que a pasta atual é a raiz. Procuramos a raiz entre a pasta atual e seus diretórios-pai.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

PASTA_ATUAL = Path.cwd().resolve()

def encontrar_raiz(inicio: Path):
    candidatos = [inicio, *inicio.parents]
    # Primeiro, procurar marcadores fortes da raiz do repositório.
    for pasta in candidatos:
        if (pasta / ".git").exists() or (
            (pasta / "documentos").exists() and
            (pasta / "tarefas").exists()
        ):
            return pasta
    # Fallback para a estrutura conhecida deste projeto.
    for pasta in candidatos:
        if pasta.name.lower() == "icmpmaster":
            return pasta
    return None

RAIZ = encontrar_raiz(PASTA_ATUAL)
if RAIZ is None:
    RAIZ = PASTA_ATUAL

print("Pasta atual do notebook:", PASTA_ATUAL)
print("Raiz estimada do projeto:", RAIZ)
print("A raiz existe:", RAIZ.exists())

## 1. Localizar os arquivos de entrada

Entradas preferidas:
- `data/interim/periodo_a_bruto.csv`
- `data/interim/periodo_b_bruto.csv`

Como alternativa, o notebook procura esses nomes dentro do projeto. Também procura um CSV bruto único, mas **não separa automaticamente A/B sem limites de tempo confirmados**.

In [ ]:
def procurar_arquivo(nome: str, raiz: Path):
    # Evita pesquisar diretórios muito pesados como .git e ambientes virtuais.
    ignorar = {".git", ".venv", "venv", "node_modules", "__pycache__"}
    for pasta, dirs, arquivos in __import__("os").walk(raiz):
        dirs[:] = [d for d in dirs if d not in ignorar]
        if nome in arquivos:
            return Path(pasta) / nome
    return None

A_CANDIDATOS = [
    RAIZ / "data" / "interim" / "periodo_a_bruto.csv",
    RAIZ / "data" / "raw" / "periodo_a_bruto.csv",
]
B_CANDIDATOS = [
    RAIZ / "data" / "interim" / "periodo_b_bruto.csv",
    RAIZ / "data" / "raw" / "periodo_b_bruto.csv",
]

A_PATH = next((p for p in A_CANDIDATOS if p.is_file()), None) or procurar_arquivo("periodo_a_bruto.csv", RAIZ)
B_PATH = next((p for p in B_CANDIDATOS if p.is_file()), None) or procurar_arquivo("periodo_b_bruto.csv", RAIZ)
ALL_PATH = (
    (RAIZ / "data" / "interim" / "medicoes_brutas_auditadas.csv")
    if (RAIZ / "data" / "interim" / "medicoes_brutas_auditadas.csv").is_file()
    else procurar_arquivo("medicoes_brutas_auditadas.csv", RAIZ)
)

print("CSV do período A:", A_PATH if A_PATH else "não encontrado")
print("CSV do período B:", B_PATH if B_PATH else "não encontrado")
print("CSV bruto único:", ALL_PATH if ALL_PATH else "não encontrado")

DADOS_CARREGADOS = False
A = None
B = None

if A_PATH is not None and B_PATH is not None:
    A = pd.read_csv(A_PATH)
    B = pd.read_csv(B_PATH)
    DADOS_CARREGADOS = True
    print("\nArquivos A e B carregados.")
elif ALL_PATH is not None:
    print("\nFoi encontrado um CSV único, mas ainda é necessário separar A e B com os limites oficiais.")
    print("Não vou inventar datas nem misturar os períodos. Confira a configuração da coleta e execute a Tarefa 1.")
else:
    print("\nAinda não encontrei os CSVs de entrada da Tarefa 1.")
    print("Verifique se a coleta foi executada e se os arquivos foram salvos em data/interim/.")
    print("Pastas pesquisadas a partir de:", RAIZ)

    # Mostrar arquivos de coleta próximos, para ajudar no diagnóstico.
    relevantes = []
    for pasta, dirs, arquivos in __import__("os").walk(RAIZ):
        dirs[:] = [d for d in dirs if d not in {".git", ".venv", "venv", "node_modules", "__pycache__"}]
        for nome in arquivos:
            if nome.lower().endswith((".csv", ".json")) and (
                "coleta" in nome.lower() or "periodo" in nome.lower()
                or "medicoes" in nome.lower() or "medicoes" in nome.lower()
            ):
                relevantes.append(Path(pasta) / nome)
    if relevantes:
        print("\nArquivos possivelmente relacionados encontrados:")
        for p in relevantes[:80]:
            print(" -", p)
    else:
        print("Nenhum CSV/JSON com nome relacionado à coleta foi localizado.")

## 2. Validar estrutura e auditar os períodos

Esta célula só executa a auditoria quando A e B foram carregados. Se os arquivos estiverem ausentes, ela imprime a orientação e segue sem criar variáveis incompletas.

In [ ]:
if not DADOS_CARREGADOS:
    print("Etapa ignorada: primeiro precisamos localizar ou gerar os arquivos A e B na Tarefa 1.")
else:
    obrigatorias = {"timestamp", "fluxo_id", "latencia_ms"}
    erros = []
    for nome, df in (("A", A), ("B", B)):
        faltantes = obrigatorias - set(df.columns)
        if faltantes:
            erros.append(f"Período {nome}: faltam colunas {sorted(faltantes)}")
    if erros:
        print("Os arquivos foram encontrados, mas o formato não corresponde ao esperado:")
        for erro in erros:
            print(" -", erro)
        DADOS_CARREGADOS = False
        A = None
        B = None
    else:
        for nome, df in (("A", A), ("B", B)):
            df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")
            for col in ["latencia_ms", "perda_pct", "jitter_ms", "sent", "rcvd", "timeout_atual"]:
                if col in df.columns:
                    df[col] = pd.to_numeric(df[col], errors="coerce")
            print(
                f"Período {nome}: {len(df):,} linhas | "
                f"{df['fluxo_id'].nunique(dropna=True):,} fluxos | "
                f"RTT ausente: {df['latencia_ms'].isna().sum():,}"
            )
        print("Duplicatas A (timestamp + fluxo):", int(A.duplicated(["timestamp", "fluxo_id"]).sum()))
        print("Duplicatas B (timestamp + fluxo):", int(B.duplicated(["timestamp", "fluxo_id"]).sum()))
        intersecao = set(zip(A["timestamp"], A["fluxo_id"])) & set(zip(B["timestamp"], B["fluxo_id"]))
        print("Pares timestamp + fluxo presentes em A e B:", len(intersecao))

## 3. Preparar campos brutos

A perda é calculada a partir de `sent` e `rcvd` quando `perda_pct` não existe. Timeout é mantido como evento. RTT ausente continua ausente — nunca é convertido para zero.

In [ ]:
def preparar_bruto(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "sent" in df.columns and "rcvd" in df.columns:
        sent = pd.to_numeric(df["sent"], errors="coerce")
        rcvd = pd.to_numeric(df["rcvd"], errors="coerce")
        perda_calculada = np.where(sent > 0, (sent - rcvd) / sent * 100, np.nan)
        if "perda_pct" not in df.columns:
            df["perda_pct"] = perda_calculada
        else:
            df["perda_pct"] = pd.to_numeric(df["perda_pct"], errors="coerce").fillna(
                pd.Series(perda_calculada, index=df.index)
            )
    elif "perda_pct" not in df.columns:
        df["perda_pct"] = np.nan

    if "jitter_ms" not in df.columns:
        df["jitter_ms"] = np.nan
    if "timeout_atual" not in df.columns:
        df["timeout_atual"] = (
            df["latencia_ms"].isna() | (df["perda_pct"] >= 100)
        ).astype(int)
    else:
        df["timeout_atual"] = pd.to_numeric(df["timeout_atual"], errors="coerce")
        df["timeout_atual"] = df["timeout_atual"].fillna(
            (df["latencia_ms"].isna() | (df["perda_pct"] >= 100)).astype(int)
        )
    # Nunca transformar RTT ausente em 0.
    df.loc[df["latencia_ms"].isna(), "latencia_ms"] = np.nan
    return df

if not DADOS_CARREGADOS:
    print("Etapa ignorada: dados A/B ainda não estão disponíveis.")
else:
    A = preparar_bruto(A)
    B = preparar_bruto(B)
    print("Campos brutos verificados; RTT ausente permanece NaN.")

## 4. Construir baseline por fluxo usando somente A

Regra do projeto: pelo menos **1.500 RTT válidos** no Período A. Fluxos abaixo desse piso recebem status `baseline_insuficiente`; não recebem rótulo no Período B.

In [ ]:
MIN_RTT_BASELINE = 1500

if not DADOS_CARREGADOS:
    print("Etapa ignorada: dados A/B ainda não estão disponíveis.")
else:
    fichas = []
    for fluxo, grupo in A.sort_values("timestamp").groupby("fluxo_id", dropna=True):
        rtt = pd.to_numeric(grupo["latencia_ms"], errors="coerce").dropna()
        n_validos = int(rtt.size)
        mediana = float(rtt.median()) if n_validos >= MIN_RTT_BASELINE else np.nan
        mad = float((rtt - rtt.median()).abs().median()) if n_validos >= MIN_RTT_BASELINE else np.nan
        q1 = float(rtt.quantile(0.25)) if n_validos >= MIN_RTT_BASELINE else np.nan
        q3 = float(rtt.quantile(0.75)) if n_validos >= MIN_RTT_BASELINE else np.nan
        jitter = pd.to_numeric(grupo["jitter_ms"], errors="coerce").dropna()
        perda = pd.to_numeric(grupo["perda_pct"], errors="coerce").dropna()
        fichas.append({
            "fluxo_id": fluxo,
            "n_medicoes_A": int(len(grupo)),
            "n_rtt_validos_A": n_validos,
            "status_baseline": "ok" if n_validos >= MIN_RTT_BASELINE else "baseline_insuficiente",
            "mediana": mediana,
            "MAD": mad,
            "IQR": (q3 - q1) if n_validos >= MIN_RTT_BASELINE else np.nan,
            "jitter_tipico": float(jitter.median()) if len(jitter) else np.nan,
            "perda_tipica": float(perda.median()) if len(perda) else np.nan,
            "prop_resposta": n_validos / len(grupo) if len(grupo) else np.nan,
        })
    baseline = pd.DataFrame(fichas)
    print("Fichas de baseline:", len(baseline))
    print(baseline["status_baseline"].value_counts(dropna=False).to_string())
    display(baseline.head())

## 5. Calcular métricas e rótulos no Período B

Critérios aplicados na ordem definida na tarefa:
- **FALHA** se `perda_pct >= 10`, `n5_timeout >= 3`, `z_robusto >= 3.5` ou `n5_aumento80 >= 2`.
- **RISCO** se `n5_risco >= 2` e, na medição atual, ocorrer pelo menos um: `2 <= z_robusto < 3.5`, `30 <= aumento_pct <= 80` ou `jitter_relativo >= 3`.
- Caso contrário, **OK**.

As janelas móveis usam as últimas cinco medições do mesmo fluxo, incluindo a atual. Fluxos sem baseline suficiente ficam sem classe.

In [ ]:
if not DADOS_CARREGADOS:
    print("Etapa ignorada: dados A/B ainda não estão disponíveis.")
else:
    dados_B = B.merge(baseline, on="fluxo_id", how="left", validate="many_to_one")
    dados_B["baseline_ok"] = dados_B["status_baseline"].eq("ok")
    dados_B["timeout_atual"] = dados_B["timeout_atual"].fillna(
        (dados_B["latencia_ms"].isna() | (dados_B["perda_pct"] >= 100)).astype(int)
    ).astype(int)

    # Métricas relativas, só quando há baseline suficiente e RTT atual válido.
    denominador = 1.4826 * dados_B["MAD"]
    alternativa = np.maximum(dados_B["IQR"] / 1.349, 1.0)
    denominador_final = np.where(denominador > 0, denominador, alternativa)
    dados_B["z_robusto"] = np.nan
    tem_rtt = dados_B["latencia_ms"].notna() & dados_B["baseline_ok"]
    mesma_mediana_mad_zero = tem_rtt & dados_B["MAD"].eq(0) & dados_B["latencia_ms"].eq(dados_B["mediana"])
    dados_B.loc[mesma_mediana_mad_zero, "z_robusto"] = 0.0
    calcula_z = tem_rtt & ~mesma_mediana_mad_zero
    dados_B.loc[calcula_z, "z_robusto"] = (
        (dados_B.loc[calcula_z, "latencia_ms"] - dados_B.loc[calcula_z, "mediana"]) /
        pd.Series(denominador_final, index=dados_B.index).loc[calcula_z].replace(0, 1.0)
    )

    dados_B["aumento_pct"] = np.where(
        tem_rtt & (dados_B["mediana"] > 0),
        (dados_B["latencia_ms"] - dados_B["mediana"]) / dados_B["mediana"] * 100,
        np.nan
    )
    dados_B["jitter_relativo"] = np.nan
    tem_jitter = dados_B["jitter_ms"].notna() & dados_B["baseline_ok"]
    base_jitter_zero = tem_jitter & dados_B["jitter_tipico"].eq(0)
    dados_B.loc[base_jitter_zero & dados_B["jitter_ms"].eq(0), "jitter_relativo"] = 1.0
    dados_B.loc[base_jitter_zero & (dados_B["jitter_ms"] > 0), "jitter_relativo"] = 3.0
    calcula_jitter = tem_jitter & (dados_B["jitter_tipico"] > 0)
    dados_B.loc[calcula_jitter, "jitter_relativo"] = (
        dados_B.loc[calcula_jitter, "jitter_ms"] / dados_B.loc[calcula_jitter, "jitter_tipico"]
    )

    # Critérios por medição. Para n5_risco, conta as medições que cumprem
    # ao menos um dos critérios individuais da linha RISCO (antes de exigir n5_risco >= 2).
    dados_B["criterio_risco_atual"] = (
        dados_B["z_robusto"].ge(2) & dados_B["z_robusto"].lt(3.5)
    ) | dados_B["aumento_pct"].between(30, 80, inclusive="both") | dados_B["jitter_relativo"].ge(3)

    dados_B = dados_B.sort_values(["fluxo_id", "timestamp"]).reset_index(drop=True)
    agrupado = dados_B.groupby("fluxo_id", sort=False)
    dados_B["n5_timeout"] = agrupado["timeout_atual"].transform(lambda s: s.rolling(5, min_periods=1).sum())
    dados_B["n5_aumento80"] = agrupado["aumento_pct"].transform(lambda s: s.gt(80).rolling(5, min_periods=1).sum())
    dados_B["n5_risco"] = agrupado["criterio_risco_atual"].transform(lambda s: s.rolling(5, min_periods=1).sum())

    dados_B["classe"] = pd.NA
    elegiveis = dados_B["baseline_ok"]
    falha = (
        dados_B["perda_pct"].ge(10) |
        dados_B["n5_timeout"].ge(3) |
        dados_B["z_robusto"].ge(3.5) |
        dados_B["n5_aumento80"].ge(2)
    )
    risco = (
        dados_B["n5_risco"].ge(2) &
        dados_B["criterio_risco_atual"]
    )
    dados_B.loc[elegiveis & ~falha & ~risco, "classe"] = "OK"
    dados_B.loc[elegiveis & ~falha & risco, "classe"] = "RISCO"
    dados_B.loc[elegiveis & falha, "classe"] = "FALHA"

    print("Linhas em B:", len(dados_B))
    print("Linhas sem classe (baseline insuficiente):", int(dados_B["classe"].isna().sum()))
    print("\nDistribuição de classes:")
    print(dados_B["classe"].value_counts(dropna=False).to_string())
    display(dados_B.head())

## 6. Recorte temporal dentro do Período B

A divisão é feita por fluxo e por ordem temporal: os 50% mais antigos para treino, os 20% seguintes para validação e os 30% mais recentes para teste. Nenhuma linha do Período A entra nesses conjuntos.

In [ ]:
if not DADOS_CARREGADOS:
    print("Etapa ignorada: dados A/B ainda não estão disponíveis.")
else:
    partes = []
    dados_rotulados = dados_B[dados_B["classe"].notna()].copy()
    for fluxo, grupo in dados_rotulados.sort_values("timestamp").groupby("fluxo_id", sort=False):
        grupo = grupo.sort_values("timestamp").copy()
        n = len(grupo)
        if n < 3:
            grupo["particao"] = "treino"
        else:
            corte_treino = max(1, int(np.floor(n * 0.50)))
            corte_validacao = max(corte_treino + 1, int(np.floor(n * 0.70)))
            corte_validacao = min(corte_validacao, n - 1)
            grupo["particao"] = "teste"
            grupo.iloc[:corte_treino, grupo.columns.get_loc("particao")] = "treino"
            grupo.iloc[corte_treino:corte_validacao, grupo.columns.get_loc("particao")] = "validacao"
        partes.append(grupo)
    dataset_rotulado = pd.concat(partes, ignore_index=True) if partes else pd.DataFrame()
    if not dataset_rotulado.empty:
        print("Contagem por partição:")
        print(dataset_rotulado["particao"].value_counts().to_string())
        print("\nClasses por partição:")
        display(pd.crosstab(dataset_rotulado["particao"], dataset_rotulado["classe"]))
    else:
        print("Não há linhas rotuladas para dividir.")

## 7. Exportar artefatos

Os arquivos são gravados em `data/processed/`. O bruto original não é sobrescrito. Se os dados de entrada não estiverem disponíveis, a exportação será ignorada com uma mensagem clara.

In [ ]:
if not DADOS_CARREGADOS:
    print("Exportação ignorada: primeiro gere/ubique os CSVs da Tarefa 1.")
else:
    SAIDA = RAIZ / "data" / "processed"
    SAIDA.mkdir(parents=True, exist_ok=True)
    caminho_baseline = SAIDA / "baseline_por_fluxo.csv"
    caminho_rotulado = SAIDA / "dataset_rotulado_B.csv"
    baseline.to_csv(caminho_baseline, index=False)
    dados_B.to_csv(caminho_rotulado, index=False)
    caminho_particoes = SAIDA / "dataset_rotulado_B_com_particao.csv"
    dataset_rotulado.to_csv(caminho_particoes, index=False)
    print("Arquivos exportados:")
    print("-", caminho_baseline)
    print("-", caminho_rotulado)
    print("-", caminho_particoes)

## 8. Checklist de validação e diário

Antes de seguir para a Tarefa 3, registre as contagens, verifique exemplos de fluxo curto e longo e confirme que nenhum RTT ausente virou zero.

- [ ] Os arquivos de A e B vieram da Tarefa 1.
- [ ] A e B não compartilham pares `timestamp + fluxo_id`.
- [ ] A baseline usa somente A.
- [ ] Fluxos com menos de 1.500 RTT válidos não recebem classe.
- [ ] FALHA tem prioridade sobre RISCO.
- [ ] A divisão temporal usa somente B.
- [ ] País, IP, rota e `fluxo_id` não devem ser entradas da árvore na Tarefa 3.

**Limitação conhecida:** este notebook só consegue processar automaticamente CSVs que tenham as colunas `timestamp`, `fluxo_id` e `latencia_ms`, além dos campos brutos opcionais indicados. Se a Tarefa 1 tiver um esquema diferente, é preciso ajustar a normalização de entrada antes de interpretar os resultados.